In [1]:
import numpy as np
import pymdp
# from pymdp.inference import run_vanilla_fpi
from pymdp.algos import run_vanilla_fpi
from pymdp.utils import obj_array_uniform
from pymdp.maths import softmax, softmax_obj_arr
from pymdp.agent import Agent

In [2]:
# Hidden State Factor 1 = Disease {Diseased, Not Diseased}
# Observation Modality 1 = Test {Positive, Negative}
# No transitions, no preferences => B, C not neeeded

hidden_states = ['Diseased', 'Not Diseased']
test_observations = ['Positive', 'Negative']

# Priors about hidden states => D matrix [D, ND]
# Assuming 1% prevalance 
p_disease = 0.01
D = np.array([p_disease, 1-p_disease])

D_mtx = obj_array_uniform([D.shape])
D_mtx[0] = D.copy()

D_mtx

array([array([0.01, 0.99])], dtype=object)

In [3]:
# B -> (states, states, actions)
B = np.eye(len(hidden_states))
B = B[:, :, np.newaxis] # Add one dim for actions, even tho no actions

B_mtx = obj_array_uniform([B.shape])
B_mtx[0] = B.copy()

B_mtx

array([array([[[1.],
               [0.]],

              [[0.],
               [1.]]])], dtype=object)

In [5]:
# Likelihoods => A matrix 2x2 (obs x hidden states)
# P(diseased | positive) {sensitivity/TP rate} = 0.99
# P(not diseased | positive) {FP rate} = 0.05
import numpy as np

tp_rate = 0.99 # D,P
fp_rate = 0.05 # ND, P
tn_rate = 1-fp_rate # ND, N
fn_rate = 1-tp_rate # D, N
A = np.array([[tp_rate, fp_rate], [fn_rate, tn_rate]])

A_mtx = np.array([A], dtype=object)

len(A_mtx)
A_mtx[0].dtype

dtype('O')

In [5]:
obs = [[1,0], [0,1]] # positive, negative

qs = run_vanilla_fpi(A_mtx, np.array([1, 0]), num_obs=[2], num_states=[2], prior=D_mtx)
print(qs)

[array([0.16666667, 0.83333333])]


In [8]:
agent = Agent(A=A, B=B_mtx, D=D_mtx)
agent.infer_states(np.array([0]))

array([array([0.16666667, 0.83333333])], dtype=object)